# Basic LLM-powered Code Generation and Benchmark

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import time

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')  

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:8]}") 
else:
    print("OpenAI API Key not set")

In [ ]:
openai = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.getenv('OPENROUTER_API_KEY'), 
)
MODEL = 'openai/gpt-4.1-mini'

In [ ]:
# Define your coding problem
problem_prompt = "Write Python code to solve the maximum subarray sum problem using Kadane's algorithm."


In [ ]:
# Function to prompt an LLM and get Python code
def get_python_code_from_llm(prompt, model="gpt-4"):
    response = openai.ChatCompletion.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


In [ ]:
# Get python code from the LLM
python_code = get_python_code_from_llm(problem_prompt) 

In [ ]:
# Extract function definition from LLM output (simple way)
def extract_and_run_code(code, test_case):
    # Save the code to a file and run it as a module
    code_file = "llm_code.py"
    with open(code_file, "w") as f:
        f.write(code)
        f.write(f"\nprint(max_subarray_sum({test_case}))\n")  # Add test case

    start = time.time()
    try:
        # Run the file and capture output
        result = os.popen(f"python {code_file}").read()
    except Exception as e:
        result = f"Error: {e}"
    end = time.time()
    os.remove(code_file)
    return result.strip(), end - start

In [ ]:
# Test case for Kadane's algorithm
test_case = [-2, 1, -3, 4, -1, 2, 1, -5, 4]

In [ ]:
# Run generated code (benchmark)
import os
output, elapsed = extract_and_run_code(python_code, test_case)
print("\nResult:", output)
print("Elapsed time:", elapsed, "seconds")

# Gradio UI Sample

In [ ]:

def llm_generate_and_run(problem_text, model_name):
    code = get_python_code_from_llm(problem_text, model=model_name)
    output, elapsed = extract_and_run_code(code, test_case)
    return f"{code}\n\nResult: {output}\nElapsed: {elapsed:.5f} sec"

gr.Interface(
    fn=llm_generate_and_run,
    inputs=[
        gr.Textbox(label="Coding Problem", value=problem_prompt),
        gr.Dropdown(["gpt-4", "gpt-3.5-turbo"], label="Model", value="gpt-4"),
    ],
    outputs=gr.Textbox(label="Code + Benchmark"),
).launch()